# Deploying Gemma 4 MoE with TensorRT-LLM (AutoDeploy)

This notebook walks you through serving **Gemma 4** (26B total, 4B activated MoE) with TensorRT-LLM using the **AutoDeploy** backend—same pattern as the Mistral Small 4 and GLM-4.7-Flash cookbooks in this folder.

[TensorRT-LLM](https://nvidia.github.io/TensorRT-LLM/) is NVIDIA's open-source library for accelerating LLM inference on NVIDIA GPUs. AutoDeploy uses Hugging Face `transformers` modeling code and TensorRT-LLM graph transforms. See the [AutoDeploy guide](https://nvidia.github.io/TensorRT-LLM/torch/auto_deploy/auto-deploy.html).

**Model resources:**
- [Gemma 4 collection (Hugging Face)](https://huggingface.co/collections/google/gemma-4)
- Instruction-tuned MoE: [`google/gemma-4-26B-A4B-it`](https://huggingface.co/google/gemma-4-26B-A4B-it)
- Base MoE (no chat template): [`google/gemma-4-26B-A4B`](https://huggingface.co/google/gemma-4-26B-A4B)

**Bundled AutoDeploy YAML (this branch):**
- **Instruction:** `examples/auto_deploy/model_registry/configs/gemma4_moe.yaml` — text-only export path; `attn_backend: triton_paged` (head_dim 512 / paged KV, CUDA-graph friendly).
- **Base:** `examples/auto_deploy/model_registry/configs/gemma4_moe_base.yaml` — same stack for the base checkpoint.

`trtllm-serve` takes **one** YAML path via `--extra_llm_api_options` (or `--config`). The bundled MoE YAMLs omit `world_size`; add it (or copy the YAML and edit) so it matches your GPU count when you use tensor parallel or multi-GPU loading.


## Prerequisites and environment

Run TensorRT-LLM in a GPU container, for example:

```shell
docker run --rm -it --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 --gpus=all -p 8000:8000 nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc1
```

Use a TensorRT-LLM checkout that includes Gemma 4 AutoDeploy support (model card, tokenizer, and any required bridges should match your branch).


In [ ]:
# If pip is not available
!python -m ensurepip --default-pip

In [ ]:
%pip install torch openai

## Verify GPU

Confirm CUDA and visible devices before starting the server.


In [1]:
import sys

import torch

print(f"Python: {sys.version}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

Python: 3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]
CUDA available: True
Num GPUs: 8
GPU[0]: NVIDIA H100 80GB HBM3
GPU[1]: NVIDIA H100 80GB HBM3
GPU[2]: NVIDIA H100 80GB HBM3
GPU[3]: NVIDIA H100 80GB HBM3
GPU[4]: NVIDIA H100 80GB HBM3
GPU[5]: NVIDIA H100 80GB HBM3
GPU[6]: NVIDIA H100 80GB HBM3
GPU[7]: NVIDIA H100 80GB HBM3


## OpenAI-compatible server

From a shell **inside** the container, at the TensorRT-LLM repo root, start `trtllm-serve` with AutoDeploy.

Use the Gemma 4 MoE YAML under `examples/auto_deploy/model_registry/configs/` (see the introduction). Add `world_size` to that YAML if your serve command needs an explicit tensor-parallel device count.


### Load the model

**Instruction-tuned (`gemma4_moe.yaml`):**

```shell
trtllm-serve "google/gemma-4-26B-A4B-it" \
  --host 0.0.0.0 \
  --port 8000 \
  --backend _autodeploy \
  --trust_remote_code \
  --extra_llm_api_options examples/auto_deploy/model_registry/configs/gemma4_moe.yaml
```

**Base checkpoint:** use model id `google/gemma-4-26B-A4B` and `examples/auto_deploy/model_registry/configs/gemma4_moe_base.yaml` instead.


When the server finishes loading weights, it is ready for requests.


## Use the API

Send chat completions with the OpenAI Python client pointed at the local server.


In [2]:
from openai import OpenAI

BASE_URL = "http://0.0.0.0:8000/v1"
API_KEY = "null"
MODEL_ID = "google/gemma-4-26B-A4B-it"

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

In [3]:
# Basic chat completion
print("Chat completion example")
print("=" * 50)

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is 15% of 85? Show your reasoning."},
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
)

print("Response:")
print(response.choices[0].message.content)

Chat completion example
Response:
To find 15% of 85, you can use a few different methods. Here are two easy ways to think about it:

### Method 1: The Breakdown Method (Mental Math)
This is often the easiest way to calculate percentages in your head by breaking the percentage into manageable parts (10% and 5%).

1.  **Find 10% of 85:**
    To find 10%, simply move the decimal point one place to the left.
    $85 \div 10 = 8.5$
2.  **Find 5% of 85:**
    Since 5% is half of 10%, just divide your previous answer by 2.
    $8.5 \div 2 = 4.25$
3.  **Add them together:**
    $10\% + 5\% = 15\%$
    $8.5 + 4.25 = 12.75$

***

### Method 2: The Multiplication Method (Calculator/Paper)
To find a percentage, you can convert the percentage into a decimal and multiply it by the total number.

1.  **Convert 15% to a decimal:**
    $15\% = \frac{15}{100} = 0.15$
2.  **Multiply by 85:**
    $85 \times 0.15 = 12.75$

**Final Answer:**
15% of 85 is **12.75**.


In [4]:
# Streaming chat completion
print("Streaming response:")
print("=" * 50)

stream = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"},
    ],
    temperature=0.7,
    max_tokens=1024,
    stream=True,
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

Streaming response:
The first 5 prime numbers are **2, 3, 5, 7, and 11**.

## Additional resources

- [Gemma 4 collection (Hugging Face)](https://huggingface.co/collections/google/gemma-4)
- [TensorRT-LLM documentation](https://nvidia.github.io/TensorRT-LLM/)
- [AutoDeploy guide](https://nvidia.github.io/TensorRT-LLM/torch/auto_deploy/auto-deploy.html)
- [`gemma4_moe.yaml`](../model_registry/configs/gemma4_moe.yaml), [`gemma4_moe_base.yaml`](../model_registry/configs/gemma4_moe_base.yaml), [`models.yaml`](../model_registry/models.yaml)
